# Camillion — One-Click Proofs

**How to use:** set `REPO_URL` in the first cell, then **Runtime to Run all**. Read the green `PASS` lines and the final `VERDICT`. No data upload, no path editing.

This runs the repo's 56 unit tests plus the 6 PyTorch/SB3 proofs the sandbox couldn't run (check_env, single-batch overfit with VecNormalize, PPO-update sanity, save/load + add-alpha stability).

In [ ]:
# ====================================================================
#                 >>> ONLY CHANGE REPO_URL <<<
# ====================================================================
# === SETUP — runs first on "Run all". Set REPO_URL once; never touch paths again. ===
REPO_URL = "https://github.com/monty313/Camillion_Claude_RL_model.git"   # already your repo — just Run all

import os, sys, subprocess
if "YOUR_USERNAME" in REPO_URL:
    raise SystemExit("Set REPO_URL (top of this cell) to your GitHub repo, then Runtime > Run all.")
REPO = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(REPO) and os.path.basename(os.getcwd()) != REPO:
    print("cloning", REPO_URL, "..."); subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print("installing deps (quiet, ~1-2 min first run)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "stable-baselines3[extra]", "gymnasium", "torch", "pandas", "numpy"], check=True)
print("setup done. working dir:", os.getcwd())

In [ ]:
# ============ CAMILLION — ONE-CLICK PROOFS  (just read the green checks) ============
import numpy as np, pandas as pd, sys, tempfile, os, warnings
warnings.filterwarnings("ignore")
results = []
def check(name, fn):
    try:
        msg = fn(); results.append((name, True)); print(f"PASS  {name}\n      {msg}\n")
    except Exception as e:
        results.append((name, False)); print(f"FAIL  {name}\n      {type(e).__name__}: {e}\n")

# ---- synthetic leak-safe cache: no data upload, fully self-contained ----
from src.data.cache_builder import build_aligned_indicators
def make_cache(n=4000, seed=1):
    rng = np.random.default_rng(seed); idx = pd.date_range("2026-01-01", periods=n, freq="1min")
    close = 100 + np.cumsum(rng.standard_normal(n) * 0.04 + 0.005)
    df = pd.DataFrame({"open": close, "high": close+.03, "low": close-.03, "close": close, "volume": 1.}, index=idx)
    return build_aligned_indicators(df), df["close"].values.astype("float32"), df.index.values.astype("datetime64[ns]").astype("int64")
IND, CL, TN = make_cache()

from src.strategies.registry import AlphaRegistry
from src.strategies.gravity_30m_4h_alpha import register_gravity_alpha
def reg_factory():
    r = AlphaRegistry(); register_gravity_alpha(r); return r   # gravity alpha, family mode (default)

# ---- P1: the repo's own unit suite (no torch) ----
def p1():
    import subprocess
    out = subprocess.run([sys.executable, "tools/run_tests.py"], capture_output=True, text=True)
    line = [l for l in out.stdout.splitlines() if "passed" in l][-1]
    assert "0 failed" in line, out.stdout[-600:]
    return line.strip()
check("Repo unit suite (stdlib)", p1)

# ---- P2: gravity alpha wired, family mode, one slot, obs still 367 (no torch) ----
def p2():
    from src.env.trading_env import TradingEnv
    from src.strategies.gravity_30m_4h_alpha import Gravity30m4hAlpha
    reg = AlphaRegistry(); slot = register_gravity_alpha(reg)
    env = TradingEnv(IND, CL, TN, reg, warmup=210); o, _ = env.reset()
    assert o.shape == (367,) and o[264 + slot] == 1.0 and Gravity30m4hAlpha().VOTE_MODE == "family"
    return f"gravity in slot {slot}, mode=family, obs shape {o.shape}, mask bit set"
check("Gravity alpha wired (family, 1 slot, 367)", p2)

# ---- P3: SB3 contract ----
def p3():
    from stable_baselines3.common.env_checker import check_env
    from src.training.gym_adapter import make_gym_env
    check_env(make_gym_env(IND, CL, TN, reg_factory(), warmup=210))
    return "gym env passes check_env  (Box(367), Discrete(4); unbounded-obs warning is expected -> handled by VecNormalize)"
check("Gym/SB3 contract (check_env)", p3)

# ---- P4: PPO can learn — single-batch overfit WITH VecNormalize (finding F1) ----
def p4():
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    from src.training.gym_adapter import make_gym_env
    mk = lambda: make_gym_env(IND[:500], CL[:500], TN[:500], reg_factory(), warmup=20, position_size=100000.0)
    venv = VecNormalize(DummyVecEnv([mk]), norm_obs=True, norm_reward=False, clip_obs=10.)
    m = PPO("MlpPolicy", venv, n_steps=256, batch_size=64, gamma=0.997, gae_lambda=0.97,
            policy_kwargs=dict(net_arch=[256, 256, 256]), verbose=0)
    def avg(steps=400):
        venv.training = False; obs = venv.reset(); tot = 0.
        for _ in range(steps):
            a, _ = m.predict(obs, deterministic=True); obs, r, d, _ = venv.step(a); tot += float(r[0])
        venv.training = True; return tot / steps
    before = avg(); m.learn(5000); after = avg()
    assert np.isfinite(before) and np.isfinite(after)
    return f"avg step reward before={before:+.4f} -> after={after:+.4f}  (PPO.learn ran finite, no NaN)"
check("PPO can learn (overfit + VecNormalize)", p4)

# ---- P5: PPO update numerically sane ----
def p5():
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    from src.training.gym_adapter import make_gym_env
    venv = VecNormalize(DummyVecEnv([lambda: make_gym_env(IND, CL, TN, reg_factory(), warmup=210, position_size=100000.0)]),
                        norm_obs=True, norm_reward=False, clip_obs=10.)
    m = PPO("MlpPolicy", venv, n_steps=512, batch_size=128, gamma=0.997, gae_lambda=0.97,
            verbose=0, policy_kwargs=dict(net_arch=[256, 256, 256]))
    m.learn(1536)
    v = m.logger.name_to_value
    pl, vl, en, kl = (float(v.get("train/policy_gradient_loss", 0)), float(v.get("train/value_loss", 0)),
                      float(v.get("train/entropy_loss", 0)), float(v.get("train/approx_kl", 0)))
    assert all(np.isfinite(x) for x in (pl, vl, en, kl))
    return f"finite losses -> policy={pl:.4f}  value={vl:.4f}  entropy={en:.4f}  approx_kl={kl:.4f}"
check("PPO update is numerically sane", p5)

# ---- P6: save / load / resume + add-alpha shape stability ----
def p6():
    from src.training import trainer
    from src.strategies.examples import register_examples
    from src.training.gym_adapter import make_gym_env
    path = os.path.join(tempfile.gettempdir(), "camillion_m")
    trainer.train(IND, CL, TN, reg_factory, total_timesteps=1536, n_envs=1, save_path=path)
    model, venv = trainer.load_for_eval(path, IND, CL, TN, reg_factory, warmup=210, position_size=100000.0)
    _ = model.predict(venv.reset(), deterministic=True)
    def reg_more():
        r = AlphaRegistry(); register_gravity_alpha(r); register_examples(r); return r   # 1 -> 4 alphas
    env2 = make_gym_env(IND, CL, TN, reg_more(), warmup=210); o2, _ = env2.reset()
    assert o2.shape == (367,)
    a, _ = model.predict(venv.normalize_obs(o2.astype(np.float32)), deterministic=True)
    return f"saved+reloaded (model+vecnorm stats); added alphas 1->4, obs still {o2.shape}, old model predicts action {int(np.ravel(a)[0])} (no shape mismatch)"
check("Save / load / resume + add-alpha stability", p6)

# ================= SUMMARY =================
ok = sum(1 for _, p in results if p); n = len(results)
print("=" * 58)
print(f"  RESULT: {ok}/{n} proofs passed")
print("=" * 58)
if ok == n:
    print("\n\u2705 GO: ready for first serious PPO run")
else:
    print("\n\u274c NO-GO: see failed section above")